# Project 27 — Heat Stress & Thermal Comfort Phenotyping with Unsupervised Learning

## Real wearable physiology + environmental clustering

This project upgrades the original K-Means exercise into a complete unsupervised-learning case study using the **Heatstroke Prevention Challenge / thermal-comfort dataset**.

The challenge data includes physiological signals/features collected with **Empatica E4 wearables**, environmental conditions, activity/task information, PMV thermal sensation, and personal thermal-assessment labels. The clustering model **does not use the personal thermal-assessment label during training**; labels are only used afterwards for interpretation and external validation.

### Business / research question

Can we discover meaningful **thermal-response phenotypes** from wearable physiology and environmental conditions without telling the algorithm the participant's subjective thermal-comfort label?

### What this notebook demonstrates

- robust data auditing and cleaning
- leakage-aware unsupervised feature selection
- missing-value handling and robust scaling
- K-Means model selection using multiple metrics
- cluster stability across random seeds
- PCA visualisation
- hierarchical clustering, Gaussian Mixtures and DBSCAN comparison
- cluster profiling and physiological interpretation
- participant/task/temperature segmentation
- external validation using labels **after** clustering
- anomaly / outlier scoring
- reusable inference pipeline
- reproducibility and integrity checks

> This is a research/portfolio analysis, not a medical diagnostic system and not a heatstroke clinical-risk score.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    normalized_mutual_info_score,
)
from sklearn.metrics import pairwise_distances
from sklearn.utils import resample

SEED = 42
RNG = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
warnings.filterwarnings("ignore")

print("Random seed:", SEED)

## 2. Load the real challenge dataset

The original notebook expected a local file called `heatstroke.csv`. This version searches sensible repository/Colab locations and fails clearly rather than silently generating synthetic data.

Download source / challenge information:
- Heatstroke Prevention Challenge: https://sites.google.com/view/heatstroke-challenge
- IEEE DataPort competition page: https://ieee-dataport.org/competitions/5th-abc-challenge-forecasting-thermal-comfort-sensations-heatstroke-prevention

In [ ]:
CANDIDATE_PATHS = [
    Path("heatstroke.csv"),
    Path("data/heatstroke.csv"),
    Path("/content/heatstroke.csv"),
    Path("/content/drive/MyDrive/heatstroke.csv"),
]

DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "heatstroke.csv was not found. Download the original Heatstroke Prevention "
        "Challenge training data, save the relevant CSV as heatstroke.csv, and rerun."
    )

raw = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Shape:", raw.shape)
display(raw.head())
display(raw.sample(min(5, len(raw)), random_state=SEED))

## 3. Standardise column names without destroying meaning

The challenge has appeared in slightly different exports. We preserve the raw frame and create a normalised working copy so the project can tolerate harmless spaces/case differences.

In [ ]:
def normalise_name(name):
    name = str(name).strip().lower()
    for ch in [" ", "-", "/", "(", ")", "[", "]", "%", "."]:
        name = name.replace(ch, "_")
    while "__" in name:
        name = name.replace("__", "_")
    return name.strip("_")

data = raw.copy()
data.columns = [normalise_name(c) for c in data.columns]

print("Normalised columns:")
print(data.columns.tolist())

## 4. Data-quality audit

In [ ]:
audit = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing": data.isna().sum(),
    "missing_pct": data.isna().mean().mul(100).round(2),
    "unique": data.nunique(dropna=True),
})
audit["constant"] = audit["unique"] <= 1

print("Rows:", len(data))
print("Columns:", data.shape[1])
print("Duplicate rows:", data.duplicated().sum())
print("Completely empty columns:", int(data.isna().all().sum()))
print("Constant columns:", int(audit["constant"].sum()))

display(audit.sort_values(["missing_pct", "unique"], ascending=[False, True]).head(30))

In [ ]:
plt.figure(figsize=(12, 5))
missing = audit["missing_pct"].sort_values(ascending=False).head(25)
missing.plot(kind="bar")
plt.ylabel("Missing values (%)")
plt.title("Top columns by missingness")
plt.tight_layout()
plt.show()

## 5. Clean categorical formatting and timestamps

A published analysis of this challenge noted that some `experiment` values contained stray whitespace. We strip categorical strings globally and derive time features when a datetime field is available.

In [ ]:
for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].astype(str).str.strip()

datetime_candidates = [c for c in data.columns if "datetime" in c or c in {"timestamp", "time"}]

for col in datetime_candidates:
    parsed = pd.to_datetime(data[col], errors="coerce")
    if parsed.notna().mean() > 0.8:
        data[f"{col}_hour"] = parsed.dt.hour + parsed.dt.minute / 60
        data[f"{col}_dayofweek"] = parsed.dt.dayofweek
        data[f"{col}_minute_of_day"] = parsed.dt.hour * 60 + parsed.dt.minute
        print(f"Derived time features from: {col}")

## 6. Identify semantic columns and prevent target leakage

The objective is **unsupervised phenotype discovery**. Therefore labels such as personal thermal assessment and PMV thermal sensation are excluded from the clustering matrix.

Identifiers such as participant ID and experiment ID are also excluded from distance calculations because arbitrary IDs should not determine physiological similarity.

In [ ]:
TARGET_PATTERNS = [
    "personal_thermal_assessment",
    "personal_thermal",
    "thermal_assessment",
]

PMV_LABEL_PATTERNS = [
    "thermal_sensation",
    "pmv_label",
    "pmv_thermal",
]

ID_PATTERNS = [
    "subject",
    "participant",
    "person_id",
    "participant_id",
    "subject_id",
]

EXPERIMENT_PATTERNS = ["experiment"]

def first_matching_column(patterns):
    for pattern in patterns:
        for col in data.columns:
            if col == pattern or pattern in col:
                return col
    return None

target_col = first_matching_column(TARGET_PATTERNS)
pmv_label_col = first_matching_column(PMV_LABEL_PATTERNS)
subject_col = first_matching_column(ID_PATTERNS)
experiment_col = first_matching_column(EXPERIMENT_PATTERNS)

print("Personal assessment label:", target_col)
print("PMV / thermal label:", pmv_label_col)
print("Participant column:", subject_col)
print("Experiment column:", experiment_col)

## 7. Build a leakage-aware numerical feature matrix

We prioritise:
- HRV features
- heart-rate features
- EDA / electrodermal features
- temperature and humidity
- accelerometer features
- age and other genuinely numeric context variables

We remove labels, arbitrary IDs, constant columns and extremely sparse columns.

In [ ]:
label_and_id_cols = {
    c for c in [target_col, pmv_label_col, subject_col, experiment_col]
    if c is not None
}

numeric_cols = data.select_dtypes(include=np.number).columns.tolist()

excluded_numeric = set()
for col in numeric_cols:
    low = col.lower()
    if any(token in low for token in [
        "label", "target", "assessment", "thermal_sensation",
        "subject", "participant", "experiment"
    ]):
        excluded_numeric.add(col)

candidate_features = [
    c for c in numeric_cols
    if c not in excluded_numeric
    and c not in label_and_id_cols
    and data[c].nunique(dropna=True) > 1
    and data[c].isna().mean() < 0.50
]

print("Candidate numeric features:", len(candidate_features))
print(candidate_features)

In [ ]:
# Remove almost-duplicate numerical features with extremely high correlation.
numeric_frame = data[candidate_features].copy()
corr = numeric_frame.corr(numeric_only=True).abs()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_to_drop = [
    column for column in upper.columns
    if any(upper[column] > 0.995)
]

features = [c for c in candidate_features if c not in high_corr_to_drop]

print("Dropped as near-duplicate (>0.995 |correlation|):", len(high_corr_to_drop))
print("Final clustering features:", len(features))

## 8. Robust preprocessing

Median imputation handles missing numeric values. Robust scaling reduces the effect of heavy-tailed HRV values and physiological outliers while keeping all rows available for clustering.

In [ ]:
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

X = preprocessor.fit_transform(data[features])

print("Feature matrix:", X.shape)
print("Finite values only:", np.isfinite(X).all())

## 9. Exploratory distributions

In [ ]:
important_patterns = [
    "temperature", "humidity", "hrv", "heart", "eda",
    "rmssd", "sdnn", "entropy", "lf", "hf"
]

important_features = []
for pattern in important_patterns:
    matches = [c for c in features if pattern in c.lower()]
    important_features.extend(matches[:2])

important_features = list(dict.fromkeys(important_features))[:12]

if important_features:
    n = len(important_features)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3.5 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, col in zip(axes, important_features):
        ax.hist(data[col].dropna(), bins=30)
        ax.set_title(col)
        ax.set_xlabel("value")
        ax.set_ylabel("count")

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No named HRV/environment feature patterns detected; using generic numerical features.")

## 10. PCA: inspect dimensional structure before clustering

PCA is used here for **visualisation and diagnostic interpretation**, not as proof that clusters exist.

In [ ]:
pca = PCA(n_components=min(10, X.shape[1]), random_state=SEED)
X_pca_full = pca.fit_transform(X)

explained = pd.DataFrame({
    "component": np.arange(1, len(pca.explained_variance_ratio_) + 1),
    "explained_variance": pca.explained_variance_ratio_,
})
explained["cumulative"] = explained["explained_variance"].cumsum()

display(explained.round(4))

plt.figure(figsize=(9, 5))
plt.plot(explained["component"], explained["cumulative"], marker="o")
plt.axhline(0.80, linestyle="--")
plt.ylabel("Cumulative explained variance")
plt.xlabel("Principal components")
plt.title("PCA cumulative explained variance")
plt.ylim(0, 1.02)
plt.grid(alpha=0.25)
plt.show()

## 11. K-Means model selection using multiple internal metrics

We do **not** choose K only from an elbow plot. For K=2…10 we compare:
- inertia (lower is better)
- silhouette score (higher is better)
- Calinski-Harabasz score (higher is better)
- Davies-Bouldin score (lower is better)

In [ ]:
k_rows = []

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=30,
        random_state=SEED,
    )
    labels = model.fit_predict(X)

    k_rows.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X, labels),
        "calinski_harabasz": calinski_harabasz_score(X, labels),
        "davies_bouldin": davies_bouldin_score(X, labels),
    })

k_eval = pd.DataFrame(k_rows)
display(k_eval.round(4))

best_k_silhouette = int(k_eval.loc[k_eval["silhouette"].idxmax(), "k"])
best_k_db = int(k_eval.loc[k_eval["davies_bouldin"].idxmin(), "k"])

print("Best K by silhouette:", best_k_silhouette)
print("Best K by Davies-Bouldin:", best_k_db)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].plot(k_eval["k"], k_eval["inertia"], marker="o")
axes[0, 0].set_title("K-Means inertia")
axes[0, 0].set_xlabel("K")

axes[0, 1].plot(k_eval["k"], k_eval["silhouette"], marker="o")
axes[0, 1].set_title("Silhouette score")
axes[0, 1].set_xlabel("K")

axes[1, 0].plot(k_eval["k"], k_eval["calinski_harabasz"], marker="o")
axes[1, 0].set_title("Calinski-Harabasz")
axes[1, 0].set_xlabel("K")

axes[1, 1].plot(k_eval["k"], k_eval["davies_bouldin"], marker="o")
axes[1, 1].set_title("Davies-Bouldin")
axes[1, 1].set_xlabel("K")

plt.tight_layout()
plt.show()

## 12. Cluster-stability analysis

A clustering solution should not collapse when the K-Means random seed changes. We quantify stability with **Adjusted Rand Index (ARI)** between the reference solution and repeated fits.

In [ ]:
SELECTED_K = best_k_silhouette

reference = KMeans(
    n_clusters=SELECTED_K,
    n_init=50,
    random_state=SEED,
).fit(X)

reference_labels = reference.labels_

stability_rows = []
for seed in range(20):
    model = KMeans(
        n_clusters=SELECTED_K,
        n_init=20,
        random_state=seed,
    ).fit(X)

    stability_rows.append({
        "seed": seed,
        "ARI_vs_reference": adjusted_rand_score(reference_labels, model.labels_),
        "inertia": model.inertia_,
    })

stability = pd.DataFrame(stability_rows)

display(stability.round(4))
print("Mean ARI stability:", round(stability["ARI_vs_reference"].mean(), 4))
print("Minimum ARI stability:", round(stability["ARI_vs_reference"].min(), 4))

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(stability["seed"], stability["ARI_vs_reference"], marker="o")
plt.ylim(-0.05, 1.05)
plt.xlabel("Random seed")
plt.ylabel("ARI vs reference clustering")
plt.title(f"K-Means stability across seeds (K={SELECTED_K})")
plt.grid(alpha=0.25)
plt.show()

## 13. Fit the final K-Means model

In [ ]:
final_kmeans = KMeans(
    n_clusters=SELECTED_K,
    init="k-means++",
    n_init=100,
    random_state=SEED,
)

cluster = final_kmeans.fit_predict(X)

results = data.copy()
results["cluster"] = cluster

print("Selected K:", SELECTED_K)
print("Final silhouette:", round(silhouette_score(X, cluster), 4))
print("Final Calinski-Harabasz:", round(calinski_harabasz_score(X, cluster), 2))
print("Final Davies-Bouldin:", round(davies_bouldin_score(X, cluster), 4))
print()
print("Cluster sizes:")
display(results["cluster"].value_counts().sort_index().rename("rows").to_frame())

## 14. PCA cluster map

In [ ]:
pca2 = PCA(n_components=2, random_state=SEED)
X2 = pca2.fit_transform(X)

plt.figure(figsize=(10, 7))
for c in sorted(np.unique(cluster)):
    mask = cluster == c
    plt.scatter(
        X2[mask, 0],
        X2[mask, 1],
        s=22,
        alpha=0.65,
        label=f"Cluster {c}",
    )

plt.xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.1%})")
plt.title("Heat-stress phenotypes in PCA space")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 15. Cluster profiles in original feature units

We compare cluster medians so HRV and environmental variables remain interpretable.

In [ ]:
profile_features = important_features if important_features else features[:12]

cluster_profile = (
    results.groupby("cluster")[profile_features]
    .median(numeric_only=True)
    .round(3)
)

display(cluster_profile)

In [ ]:
# Standardised cluster profile heatmap without seaborn.
profile_scaled = cluster_profile.copy()
for col in profile_scaled.columns:
    std = profile_scaled[col].std(ddof=0)
    if std > 0:
        profile_scaled[col] = (
            profile_scaled[col] - profile_scaled[col].mean()
        ) / std
    else:
        profile_scaled[col] = 0

fig, ax = plt.subplots(figsize=(max(10, 0.8 * len(profile_scaled.columns)), 5))
im = ax.imshow(profile_scaled.values, aspect="auto")
ax.set_xticks(range(len(profile_scaled.columns)))
ax.set_xticklabels(profile_scaled.columns, rotation=65, ha="right")
ax.set_yticks(range(len(profile_scaled.index)))
ax.set_yticklabels([f"Cluster {i}" for i in profile_scaled.index])
ax.set_title("Standardised median phenotype profiles")
plt.colorbar(im, ax=ax, label="Cluster-profile z-score")
plt.tight_layout()
plt.show()

## 16. Environmental and activity interpretation

These variables are used for **post-hoc profiling**. A cluster associated with hotter temperatures or heavier activity should not automatically be called “high heatstroke risk”; it is a discovered data segment that requires scientific/clinical validation.

In [ ]:
context_cols = []

for token in ["temperature", "humidity", "task", "activity", "age", "gender"]:
    context_cols.extend([c for c in results.columns if token in c.lower()])

context_cols = list(dict.fromkeys(context_cols))

for col in context_cols:
    print("\n###", col)
    if pd.api.types.is_numeric_dtype(results[col]):
        display(
            results.groupby("cluster")[col]
            .agg(["count", "mean", "median", "std"])
            .round(3)
        )
    else:
        display(
            pd.crosstab(
                results["cluster"],
                results[col],
                normalize="index"
            ).round(3)
        )

## 17. Participant-level cluster distribution

This checks whether a cluster is dominated by one participant rather than reflecting a repeatable phenotype across people.

In [ ]:
if subject_col is not None:
    participant_mix = pd.crosstab(
        results[subject_col],
        results["cluster"],
        normalize="index",
    )

    print("Participants:", participant_mix.shape[0])
    display(participant_mix.head(20).round(3))

    plt.figure(figsize=(12, 6))
    participant_mix.plot(
        kind="bar",
        stacked=True,
        figsize=(14, 6),
        legend=True,
    )
    plt.ylabel("Fraction of participant observations")
    plt.title("Cluster composition within each participant")
    plt.tight_layout()
    plt.show()
else:
    print("No participant/subject column was detected.")

## 18. External validation against subjective thermal assessment

This section is intentionally **after** clustering.

Adjusted Mutual Information (AMI), Normalized Mutual Information (NMI), and ARI measure association between discovered clusters and known labels. They are **not classification accuracy**, and the model was not trained to reproduce these labels.

In [ ]:
if target_col is not None:
    labelled_mask = results[target_col].notna()

    target_codes, target_names = pd.factorize(results.loc[labelled_mask, target_col])

    external = {
        "adjusted_mutual_information": adjusted_mutual_info_score(
            target_codes,
            results.loc[labelled_mask, "cluster"],
        ),
        "normalized_mutual_information": normalized_mutual_info_score(
            target_codes,
            results.loc[labelled_mask, "cluster"],
        ),
        "adjusted_rand_index": adjusted_rand_score(
            target_codes,
            results.loc[labelled_mask, "cluster"],
        ),
    }

    print(json.dumps(external, indent=2))

    label_mix = pd.crosstab(
        results.loc[labelled_mask, "cluster"],
        results.loc[labelled_mask, target_col],
        normalize="index",
    )

    display(label_mix.round(3))
else:
    print("No personal thermal-assessment label detected; external validation skipped.")

## 19. Compare different clustering families

K-Means assumes roughly spherical clusters. We compare it with:
- Agglomerative clustering
- Gaussian Mixture Models
- DBSCAN

This comparison uses internal validation only and avoids cherry-picking a method because it happens to match the subjective label.

In [ ]:
comparison_rows = []

def add_cluster_metrics(name, labels):
    labels = np.asarray(labels)
    unique = np.unique(labels)

    # Exclude DBSCAN noise (-1) for internal metric calculation.
    valid = labels != -1
    valid_labels = labels[valid]

    if valid.sum() < 3 or len(np.unique(valid_labels)) < 2:
        return

    comparison_rows.append({
        "model": name,
        "clusters": len(np.unique(valid_labels)),
        "noise_fraction": float(np.mean(labels == -1)),
        "silhouette": silhouette_score(X[valid], valid_labels),
        "calinski_harabasz": calinski_harabasz_score(X[valid], valid_labels),
        "davies_bouldin": davies_bouldin_score(X[valid], valid_labels),
    })

add_cluster_metrics("KMeans", cluster)

agg = AgglomerativeClustering(n_clusters=SELECTED_K)
agg_labels = agg.fit_predict(X)
add_cluster_metrics("Agglomerative", agg_labels)

gmm = GaussianMixture(
    n_components=SELECTED_K,
    covariance_type="full",
    random_state=SEED,
    n_init=5,
)
gmm_labels = gmm.fit_predict(X)
add_cluster_metrics("GaussianMixture", gmm_labels)

for eps in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
    db = DBSCAN(eps=eps, min_samples=max(5, int(np.sqrt(len(X)) / 3)))
    db_labels = db.fit_predict(X)
    add_cluster_metrics(f"DBSCAN eps={eps}", db_labels)

model_comparison = pd.DataFrame(comparison_rows)
display(
    model_comparison
    .sort_values("silhouette", ascending=False)
    .round(4)
)

## 20. Cluster-confidence / distance analysis

K-Means does not output probabilities, but distance to the assigned centroid is useful for:
- finding ambiguous points
- identifying unusual physiological observations
- selecting representative examples near cluster centres

In [ ]:
centroid_distances = pairwise_distances(X, final_kmeans.cluster_centers_)
assigned_distance = centroid_distances[np.arange(len(X)), cluster]

results["distance_to_centroid"] = assigned_distance
results["cluster_distance_percentile"] = (
    results.groupby("cluster")["distance_to_centroid"]
    .rank(pct=True)
)

print("Distance summary:")
display(
    results.groupby("cluster")["distance_to_centroid"]
    .describe()
    .round(3)
)

In [ ]:
# Representative observations = closest rows to each centroid.
representatives = (
    results.sort_values("distance_to_centroid")
    .groupby("cluster", as_index=False)
    .head(3)
)

show_cols = [
    c for c in [
        subject_col,
        experiment_col,
        target_col,
        pmv_label_col,
        *context_cols,
        "cluster",
        "distance_to_centroid",
    ]
    if c is not None and c in representatives.columns
]

display(representatives[show_cols].head(3 * SELECTED_K))

## 21. Potential outliers

Rows above the 99th percentile of within-cluster centroid distance are flagged for inspection. They are **not automatically bad data**; they may represent rare but legitimate physiological responses.

In [ ]:
outlier_thresholds = (
    results.groupby("cluster")["distance_to_centroid"]
    .quantile(0.99)
    .to_dict()
)

results["potential_outlier"] = [
    d > outlier_thresholds[c]
    for d, c in zip(results["distance_to_centroid"], results["cluster"])
]

print("Potential outliers:", int(results["potential_outlier"].sum()))
print("Outlier percentage:", round(results["potential_outlier"].mean() * 100, 2), "%")

display(
    results.loc[results["potential_outlier"], show_cols]
    .head(20)
)

## 22. Bootstrap stability of the selected solution

Random seeds test optimisation stability. Bootstrap resampling asks a different question: do similar cluster centres appear when the observed dataset changes slightly?

In [ ]:
bootstrap_inertias = []
bootstrap_silhouettes = []

for i in range(30):
    idx = RNG.integers(0, len(X), size=len(X))
    X_boot = X[idx]

    model = KMeans(
        n_clusters=SELECTED_K,
        n_init=20,
        random_state=SEED + i,
    ).fit(X_boot)

    labels_boot = model.labels_

    bootstrap_inertias.append(model.inertia_)
    if len(np.unique(labels_boot)) > 1:
        bootstrap_silhouettes.append(
            silhouette_score(X_boot, labels_boot)
        )

print("Bootstrap runs:", len(bootstrap_inertias))
print(
    "Bootstrap silhouette mean ± SD:",
    round(np.mean(bootstrap_silhouettes), 4),
    "±",
    round(np.std(bootstrap_silhouettes), 4),
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(bootstrap_inertias, bins=12)
axes[0].set_title("Bootstrap K-Means inertia")
axes[0].set_xlabel("Inertia")

axes[1].hist(bootstrap_silhouettes, bins=12)
axes[1].set_title("Bootstrap silhouette distribution")
axes[1].set_xlabel("Silhouette")

plt.tight_layout()
plt.show()

## 23. Reusable inference function

This turns the notebook into a small analytical component rather than a one-off experiment. New rows must contain the same selected numerical features.

In [ ]:
def assign_thermal_phenotype(new_frame):
    """
    Assign K-Means phenotype to new observations.

    Important:
    - This is an unsupervised cluster assignment, not a diagnosis.
    - Input columns must include the fitted feature set.
    """
    missing = [c for c in features if c not in new_frame.columns]
    if missing:
        raise ValueError(f"Missing required features: {missing[:10]}")

    X_new = preprocessor.transform(new_frame[features])
    labels = final_kmeans.predict(X_new)
    d = final_kmeans.transform(X_new)
    confidence_proxy = d[np.arange(len(X_new)), labels]

    return pd.DataFrame({
        "cluster": labels,
        "distance_to_centroid": confidence_proxy,
    }, index=new_frame.index)

demo_assignments = assign_thermal_phenotype(data.head(5))
display(demo_assignments)

## 24. Integrity checks

In [ ]:
assert len(cluster) == len(data)
assert X.shape[0] == len(data)
assert np.isfinite(X).all()
assert len(np.unique(cluster)) == SELECTED_K
assert set(label_and_id_cols).isdisjoint(features)
assert target_col not in features if target_col is not None else True
assert pmv_label_col not in features if pmv_label_col is not None else True
assert stability["ARI_vs_reference"].between(-1, 1).all()

print("All clustering integrity checks passed ✅")

## 25. Final project summary

This project is stronger than the original two-variable K-Means exercise because it treats clustering as a complete modelling problem:

1. **Real-world data provenance:** wearable physiology, HRV, environment, activity and subjective assessments from the Heatstroke Prevention Challenge.
2. **Leakage control:** personal thermal-assessment and PMV labels are excluded from unsupervised training.
3. **Model selection:** K is selected using silhouette, Calinski-Harabasz, Davies-Bouldin and inertia instead of a single elbow plot.
4. **Robustness:** repeated-seed and bootstrap stability checks.
5. **Model comparison:** K-Means vs hierarchical clustering vs Gaussian Mixtures vs DBSCAN.
6. **Explainability:** cluster profiles in the original feature units plus representative observations.
7. **External validation:** labels are used only after clustering via AMI/NMI/ARI—not misleading classification accuracy.
8. **Operationalisation:** reusable phenotype-assignment function and outlier monitoring.

### Interview-ready explanation

> “I used wearable HRV and environmental data to discover latent thermal-response phenotypes. I deliberately removed subjective thermal labels from model training, selected the number of clusters using multiple internal validation metrics, tested stability across seeds and bootstraps, compared alternative clustering families, then used known thermal assessments only for post-hoc external validation and interpretation.”

### Limitations

- Clusters are statistical segments, **not clinical heatstroke categories**.
- K-Means depends on the chosen feature representation and distance metric.
- Repeated measurements from the same participant introduce dependence.
- Participant-disjoint validation is required before any personalised deployment claim.
- Sensor artefacts, missingness and activity context can affect HRV-derived features.
- A future version should use the official train/test participant split and compare unsupervised phenotypes with supervised thermal-comfort forecasting.

**Suggested CV bullet:**  
Built an end-to-end unsupervised thermal-physiology analytics pipeline on wearable HRV/environment data, using robust preprocessing, K-Means model selection, PCA, cluster stability, hierarchical/GMM/DBSCAN benchmarking, phenotype profiling, external label validation and anomaly detection.